# 01. Exploring the Chactun tiles

What the inputs and labels look like before any training: value ranges, the mask
encoding, class balance on each split, and examples of each class. Figures and a
summary JSON are saved to `runs/explore/<timestamp>/`.

## Setup

On Colab this mounts Google Drive, clones the repo and installs it. Locally it assumes
`pip install -e .` has been run in the repo and the tiles are in `data/chactun/`.

In [ ]:
REPO_URL = "https://github.com/skumagai3/maya-lidar.git"
DRIVE_DIR = "/content/drive/MyDrive/maya-lidar"           # Colab: runs are saved here
DATA_ZIP = f"{DRIVE_DIR}/data/22202395.zip"               # Colab: Figshare bundle, or a zip of the tile files

import subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/maya-lidar")
    if repo.exists():
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))  # an editable install is not importable until restart
else:
    DATA_ZIP = None

import numpy as np
import matplotlib.pyplot as plt
from mayalidar import config as C, data as D, viz as V

paths = C.project_paths(DRIVE_DIR)
D.ensure_extracted(DATA_ZIP, paths.data_dir)
tiles = D.index_tiles(paths.data_dir)
print(f"{len(tiles)} tiles in {paths.data_dir}; runs go to {paths.runs_dir}")

## Splits

The official challenge split holds out tiles 1765-2093 as the test set. Train and
validation are drawn at random from tiles 0-1764 with the seed in the experiment config,
so this notebook uses the same split as `02_unet_scratch`.

In [ ]:
cfg = C.load_config("unet_scratch")
train, val, test = D.official_split(tiles, cfg["data"]["val_frac"], cfg["data"]["split_seed"])
print(f"complete tiles: {sum(t.complete for t in tiles)} of {len(tiles)}")
print(f"train {len(train)}, validation {len(val)}, test {len(test)}")
run_dir = C.new_run_dir(paths, "explore")
summary = {"n_train": len(train), "n_val": len(val), "n_test": len(test)}

## Stored values and mask encoding

The paper documents lidar tiles as 3-band uint8 and masks with 0 = feature, 255 =
background. Check both, since training hard-codes the mask value.

In [ ]:
import tifffile
raw = tifffile.imread(train[0].lidar)
print("lidar tile:", raw.shape, raw.dtype, "range", raw.min(), raw.max())
present = D.detect_mask_present_value(train)
print("value marking a feature in the masks:", present)
assert present == 0, "masks do not match the documented encoding; check before training"
summary["mask_present_value"] = present

## Class balance by split

In [ ]:
counts = {name: D.class_counts(split, present) for name, split in [("train", train), ("validation", val), ("test", test)]}
summary["class_counts"] = counts
for name, c in counts.items():
    fig = V.plot_class_counts(c, f"Chactun {name} split ({c['n_tiles']} tiles): labeled pixel fraction and tiles per class")
    V.save(fig, run_dir, f"class_counts_{name}")
    plt.show()

## Band value distributions (training split)

These are the stored 0-255 values, before standardization. Training standardizes each
band with the mean and standard deviation of its own training split.

In [ ]:
rng = np.random.default_rng(0)
sample = np.concatenate([D.read_tif(train[i].lidar).reshape(-1, 3)[::97]
                         for i in rng.choice(len(train), 200, replace=False)])
fig = V.plot_band_histograms(sample, "Stored values of the three lidar bands, 200 random training tiles")
V.save(fig, run_dir, "band_histograms_train")
plt.show()
summary["train_band_stats_200_tiles"] = {"mean": sample.mean(0).tolist(), "std": sample.std(0).tolist()}

## Examples of each class

In [ ]:
for ci, c in enumerate(D.CLASSES):
    for j in rng.permutation(len(train)):
        t = train[j]
        y = D.load_mask(t, present)
        if y[..., ci].sum() > 2000:
            fig = V.show_tile(D.read_tif(t.lidar), y, f"Training tile {t.tile_id}: lidar bands and labels (contains {c})")
            V.save(fig, run_dir, f"example_{c}_tile{t.tile_id}")
            plt.show()
            break

In [ ]:
D.save_json(summary, run_dir / "summary.json")
print("saved to", run_dir)